kernel : Python (Pixi)

In [22]:
%load_ext autoreload
%autoreload 2
import gc
import os
import pandas as pd
import scanpy as sc
from pipeline.utils.env import find_env_dir
from pipeline.utils.get_ensembl_to_symbol import get_ensg_to_symbol

raw_data_dir = find_env_dir("RAW_DATA")
pre_h5ad_dir = find_env_dir("PRE_H5AD")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
SERIES = "krzak"
SAVE_FILE = f"{SERIES}_raw.h5ad"
data_dir = os.path.join(raw_data_dir, SERIES)

adata = sc.read_h5ad(os.path.join(data_dir, "Full_cohort.h5ad"), backed="r")

In [24]:
def one_row_per_key(df: pd.DataFrame, key: str) -> pd.DataFrame:
    return df.groupby(key, sort=False, dropna=False).agg(
        lambda s: s.dropna().iloc[0] if s.dropna().nunique() == 1 else pd.NA
    )

sample_meta = one_row_per_key(
    pd.read_csv(os.path.join(data_dir, "sample_metadata.tsv"), sep="\t"),
    "sanger_sample_id",
)
sample_to_individual = one_row_per_key(
    pd.read_csv(os.path.join(data_dir, "TI_atlas-sample_individual_mapping.tsv"), sep="\t"),
    "sanger_sample_id",
)
individual_meta = pd.read_csv(
    os.path.join(data_dir, "individual_metadata.tsv"), sep="\t"
).drop_duplicates("individual_id").set_index("individual_id")

adata.obs = adata.obs.join(sample_meta, on="sanger_sample_id") #type: ignore
adata.obs = adata.obs.join(sample_to_individual, on="sanger_sample_id") #type: ignore
adata.obs = adata.obs.join(individual_meta, on="individual_id") #type: ignore

adata.obs.head() #type: ignore

,sanger_sample_id,pct_counts_gene_group__mito_transcript,n_genes_by_counts,total_counts,label,category,conf_score,biopsy_type,bead_version,bead_lot,...,chip_lot,enzyme_lot_BLP,gem_lot,individual_id,cohort,atlas,disease_status,age_binned_10,sex,ti_ses_inflamed
TCAGGTATCCGACATA-1-5892STDY10205765__41618__cellranger720_count_5892STDY10205765_GRCh38-3_0_0,5892STDY10205765,0.886711,4250,141196.0,Enterocytes progenitor crypt GPX2+,Enterocyte,0.9987180652351288,terminal ileum,V3.1,<NA>,...,lot_160047,SLCB6672,lot_155287,IBDverse-1-3A5C2FF82FA8A4,Discovery,atlas70,CD,40.0,male,inflamed
CCAATTTAGATACCAA-1-5892STDY10205765__41618__cellranger720_count_5892STDY10205765_GRCh38-3_0_0,5892STDY10205765,0.929196,4671,108158.0,Enterocytes progenitor crypt GPX2+,Enterocyte,0.9972390415425336,terminal ileum,V3.1,<NA>,...,lot_160047,SLCB6672,lot_155287,IBDverse-1-3A5C2FF82FA8A4,Discovery,atlas70,CD,40.0,male,inflamed
CATGAGTCAGGCATTT-1-5892STDY10205765__41618__cellranger720_count_5892STDY10205765_GRCh38-3_0_0,5892STDY10205765,0.686976,4169,107864.0,Enterocytes progenitor crypt GPX2+,Enterocyte,0.9770095240845306,terminal ileum,V3.1,<NA>,...,lot_160047,SLCB6672,lot_155287,IBDverse-1-3A5C2FF82FA8A4,Discovery,atlas70,CD,40.0,male,inflamed
TCTCACGGTATCTCGA-1-5892STDY10205765__41618__cellranger720_count_5892STDY10205765_GRCh38-3_0_0,5892STDY10205765,0.361118,2867,81137.0,Enterocytes progenitor crypt GPX2+,Enterocyte,0.772354776734459,terminal ileum,V3.1,<NA>,...,lot_160047,SLCB6672,lot_155287,IBDverse-1-3A5C2FF82FA8A4,Discovery,atlas70,CD,40.0,male,inflamed
TGCGGCAGTCCCAAAT-1-5892STDY10205765__41618__cellranger720_count_5892STDY10205765_GRCh38-3_0_0,5892STDY10205765,0.717362,2527,79876.0,Enterocytes progenitor crypt GPX2+,Enterocyte,0.98518622462155,terminal ileum,V3.1,<NA>,...,lot_160047,SLCB6672,lot_155287,IBDverse-1-3A5C2FF82FA8A4,Discovery,atlas70,CD,40.0,male,inflamed


In [25]:
ensg_to_symbol = get_ensg_to_symbol()

vn = pd.Series(adata.var_names, index=adata.var_names, dtype="string")
ensg = vn.str.replace(r"\.\d+$", "", regex=True)
print("versioned ENSG:", int((vn != ensg).sum()))

bad = ~ensg.str.match(r"^ENSG\d+$", na=False)
if bad.any():
    print("[WARN] Non-ENSG-like var_names examples:", vn[bad].head(20).tolist())

mapped = ensg.map(ensg_to_symbol).replace("", pd.NA).fillna(vn)

adata.var["orig_var_name"] = vn.astype(str).values
adata.var_names = mapped.astype(str).to_list()
adata.var_names_make_unique()

versioned ENSG: 0


In [26]:
assert isinstance(adata.obs, pd.DataFrame)
adata.obs["sample"] = adata.obs["sanger_sample_id"]
adata.obs["donor"] = adata.obs["individual_id"]
adata.obs["celltype_coarse"] = adata.obs["category"]
adata.obs["celltype"] = adata.obs["label"]
adata.obs["organ"] = adata.obs["biopsy_type"]
adata.obs["disease"] = adata.obs["disease_status"]
adata.obs["condition"] = adata.obs["ti_ses_inflamed"]
adata.obs["age_binned_10"] = adata.obs["age_binned_10"]
adata.obs["sex"] = adata.obs["sex"]

adata.obs["bead_lot"] = adata.obs["bead_lot"]
adata.obs["chip_lot"] = adata.obs["chip_lot"]
adata.obs["enzyme_lot"] = adata.obs["enzyme_lot_BLP"]
adata.obs["gem_lot"] = adata.obs["gem_lot"]

adata.obs["series"] = SERIES

keep = ["sample", "donor", "celltype_coarse", "celltype", "organ", "disease", "condition", "age_binned_10", "sex", "bead_lot", "chip_lot", "enzyme_lot", "gem_lot", "series"]
adata.obs.drop(columns = [c for c in adata.obs.columns if c not in keep], inplace=True) #type: ignore

In [ ]:
del adata.layers["counts"]
del adata.layers["log1p_cp10k"]

var_cols_to_drop = ["gene_group__mito_transcript", "gene_group__mito_protein", "gene_group__ribo_protein", 
                    "gene_group__ribo_rna", "n_cells_by_counts", "mean_counts", "log1p_mean_counts", 
                    "pct_dropout_by_counts", "total_counts", "log1p_total_counts"]
adata.var.drop(columns = [c for c in adata.var.columns if c in var_cols_to_drop], inplace=True) #type: ignore

In [39]:
adata.obs.index = adata.obs.index.astype(str)
adata.var.index = adata.var.index.astype(str)
adata.write_h5ad(os.path.join(pre_h5ad_dir, SAVE_FILE), compression="gzip")
del adata
gc.collect()

0